# Pendulum Agentic SysID — Analysis Notebook

Explores the identification pipeline step-by-step:
1. Plant behaviour and true parameters
2. White-box identification (NLS, no Coulomb)
3. Grey-box correction (Coulomb detection + K_c estimation)
4. Surrogate fit (GP on residual-cleaned data)
5. Comparison plots

**Run from the repo root:**  `jupyter notebook notebooks/pendulum_analysis.ipynb`  
No `ANTHROPIC_API_KEY` needed — all deterministic paths only.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)

## 1  Infrastructure setup

In [ ]:
from pathlib import Path
import tempfile

from plants.inverted_pendulum import PendulumPlant, PendulumParams
from tools.budget_manager import BudgetManager
from tools.experiment_db import ExperimentDatabase
from tools.model_registry import ModelRegistry
from tools.plant_api import PlantAPI
from core.schemas import PlantContract, SplitFlag

# Use a temp directory so the notebook is self-contained
_tmp = Path(tempfile.mkdtemp(prefix="sysid_nb_"))
print(f"Data dir: {_tmp}")

# True plant (Coulomb friction is hidden from the identification agents)
true_params = PendulumParams(J=0.05, m=0.5, L=0.30, b_v=0.02, f_c=0.08,
                              noise_std=0.001)
plant    = PendulumPlant(params=true_params, seed=42)
contract = PlantContract(
    name="pendulum_notebook",
    input_names=["torque"], output_names=["angle"],
    state_names=["theta", "theta_dot"],
    input_limits={"torque": (-2.0, 2.0)},
    sample_time=0.02,
)
budget   = BudgetManager(total=500.0)
db       = ExperimentDatabase(str(_tmp/"exp.db"), str(_tmp/"runs"))
registry = ModelRegistry(str(_tmp/"models"))
api      = PlantAPI(plant, contract, budget, db, experiment_cost=2.0)

# True physical parameters for reference
K_IN_TRUE  = 1.0 / true_params.J
TAU_D_TRUE = true_params.b_v / true_params.J
K_G_TRUE   = true_params.m * 9.81 * true_params.L / true_params.J
K_C_TRUE   = true_params.f_c / true_params.J
print(f"True params:  K_in={K_IN_TRUE:.2f}  tau_d={TAU_D_TRUE:.3f}  "
      f"K_g={K_G_TRUE:.3f}  K_c={K_C_TRUE:.3f}")

## 2  Plant exploration — free-oscillation and PRBS response

In [ ]:
from tools.solver_toolkit import generate_prbs
from agents.experiment_design import ExperimentDesignAgent

designer = ExperimentDesignAgent()

# Free oscillation (zero torque)
u_zero = lambda t: np.array([0.0])
t_free, _, y_free = plant.apply_input(u_zero, (0.0, 4.0), dt=0.02)

# PRBS excitation
seq    = designer.design_for_identification(contract, n_samples=400, seed=1)
u_func = designer.make_u_func(seq["t"], seq["u"])
t_prbs, u_prbs, y_prbs = plant.apply_input(u_func, (0.0, 8.0), dt=0.02)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(t_free, y_free[0], color="#4cc9f0", label="θ (free)")
axes[0].set(title="Free oscillation (u=0, θ₀=0.3 rad)",
             xlabel="Time [s]", ylabel="θ [rad]")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(t_prbs, y_prbs[0], color="#4cc9f0", label="θ [rad]")
ax2b = ax2.twinx()
ax2b.plot(t_prbs, u_prbs[0], color="#f72585", alpha=0.6, label="τ [N·m]")
ax2.set(title="PRBS excitation response", xlabel="Time [s]", ylabel="θ [rad]")
ax2b.set_ylabel("τ [N·m]", color="#f72585")
ax2.legend(loc="upper left"); ax2b.legend(loc="upper right")
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3  White-box identification

In [ ]:
from core.schemas import ModelArtifact, ModelType, Artifacts, Assets, Budget, Dossier, EntryPath, PhysicsAvailability
from agents.estimator import EstimatorAgent

# Inject the known ODE structure (no Coulomb term — this is what white-box gets)
WB_RHS = "K_in*tau - tau_d*theta_dot - K_g*sin(theta)"
wb_artifact = ModelArtifact(
    model_type=ModelType.WHITE_BOX,
    structure_description="θ̈ = K_in·τ − τ_d·θ̇ − K_g·sin(θ)",
    parameters={"K_in": K_IN_TRUE, "tau_d": TAU_D_TRUE, "K_g": K_G_TRUE},
    metadata={
        "normalized_rhs": WB_RHS,
        "fit_params":     ["K_in", "tau_d", "K_g"],
        "param_bounds":   {"K_in": [0, 200], "tau_d": [0, 50], "K_g": [0, 500]},
        "state_vars":     ["theta", "theta_dot"],
        "input_vars":     ["tau"],
        "output_vars":    ["theta"],
        "improvable":     True,
    },
)
wb_id = registry.store_model(wb_artifact)

# Store contract
contract_artifact = ModelArtifact(
    id=contract.id, model_type=ModelType.WHITE_BOX,
    structure_description="PlantContract",
    metadata={"plant_contract": {
        "id": contract.id, "name": contract.name,
        "input_names": contract.input_names, "output_names": contract.output_names,
        "state_names": contract.state_names,
        "input_limits": {k: list(v) for k, v in contract.input_limits.items()},
        "input_rate_limits": {}, "output_limits": {},
        "sample_time": contract.sample_time, "is_unstable": False, "description": "",
    }},
)
registry.store_model(contract_artifact)

estimator = EstimatorAgent(api, registry, db, n_samples=400)
est_report = estimator.run(wb_id, contract.id)

wb_fitted_id = est_report.metadata["model_id"]
wb_fitted    = registry.load_model(wb_fitted_id)
print("White-box fitted parameters:")
for p, v in wb_fitted.parameters.items():
    print(f"  {p:8s} = {v:.4f}")

In [ ]:
from agents.validation import ValidationAgent
from core.schemas import Verdict, VerdictResult, GapType, Rung

dossier_wb = Dossier(
    entry_path=EntryPath.WHITE_BOX,
    current_rung=Rung.WHITE,
    budget=Budget(total=500.0, spent=float(budget.spent)),
    assets=Assets(plant_contract_id=contract.id, physics=PhysicsAvailability.FULL),
    artifacts=Artifacts(current_model_id=wb_fitted_id,
                        model_history=[wb_id, wb_fitted_id],
                        dataset_ids=est_report.metadata.get("run_ids", [])),
)

val_agent = ValidationAgent(api, registry, db)
dossier_wb = val_agent(dossier_wb)

v = dossier_wb.last_verdict
print(f"White-box verdict: {v.verdict.value} / gap={v.gap_type.value}")
print(f"  RMSE:             {v.metrics.get('rmse', 'n/a'):.4f} rad")
print(f"  Coulomb corr:     {v.metrics.get('coulomb_correlation', 'n/a'):.3f}")

## 4  Grey-box correction (Coulomb detection)

In [ ]:
from agents.greybox.sub_orchestrator import GreyBoxSubOrchestrator

gb_so = GreyBoxSubOrchestrator(api, registry, db, n_samples=400)
dossier_gb = gb_so(dossier_wb)

gb_report = dossier_gb.last_report
gb_model  = registry.load_model(dossier_gb.artifacts.current_model_id)

print(f"Grey-box strategy: {gb_report.metadata.get('strategy')}")
print(f"K_c estimate:      {gb_report.metadata.get('K_c_estimate', 'n/a'):.4f}  (true={K_C_TRUE:.4f})")
print("Corrected parameters:")
for p, v_ in gb_model.parameters.items():
    print(f"  {p:8s} = {v_:.4f}")

# Validate corrected model
dossier_gb = val_agent(dossier_gb)
vg = dossier_gb.last_verdict
print(f"\nGrey-box verdict:  {vg.verdict.value} / gap={vg.gap_type.value}")
print(f"  RMSE:            {vg.metrics.get('rmse', 'n/a'):.4f} rad")

## 5  Surrogate fit (GP on accumulated data)

In [ ]:
from agents.surrogate.sub_orchestrator import SurrogateSubOrchestrator

surr_so = SurrogateSubOrchestrator(api, registry, db, n_samples=300, nn_epochs=100)
dossier_surr = surr_so(dossier_gb)

sr = dossier_surr.last_report
print(f"Surrogate model class: {sr.metadata.get('model_class')}")
print(f"Training points:       {sr.metadata.get('n_train')}")
print(f"Train RMSE:            {sr.metadata.get('train_rmse', 0):.4f} rad/s²")
print(f"Mean uncertainty std:  {sr.metadata['uncertainty'].get('mean_std', 0):.4f}")

# Validate surrogate
dossier_surr = val_agent(dossier_surr)
vs = dossier_surr.last_verdict
print(f"\nSurrogate verdict:     {vs.verdict.value} / gap={vs.gap_type.value}")
print(f"  RMSE:                {vs.metrics.get('rmse', 'n/a'):.4f} rad")

## 6  Comparison plot — true plant vs white-box vs grey-box vs surrogate

In [ ]:
from tools.symbolic_math import make_ode_simulator
from agents.validation import _make_surrogate_simulator

# Fresh validation input
val_seq = designer.design_for_validation(contract, n_samples_per_scenario=200,
                                          n_scenarios=1, seed=99)[0]
t_v  = val_seq["t"]
u_fn = designer.make_u_func(t_v, val_seq["u"])
_, u_true, y_true = plant.apply_input(u_fn, (float(t_v[0]), float(t_v[-1])), dt=0.02)
t_v = np.linspace(float(t_v[0]), float(t_v[-1]), len(y_true[0]))

def _sim_model(model_id, t, u_arr):
    m   = registry.load_model(model_id)
    rhs = m.metadata.get("normalized_rhs", "")
    if rhs == "SURROGATE":
        pred  = registry.load_object(m.metadata["surrogate_object_id"])
        sim   = _make_surrogate_simulator(pred)
        p_arr = np.array([])
    else:
        params = m.metadata.get("fit_params", [])
        state_v = m.metadata.get("state_vars", ["theta", "theta_dot"])
        input_v = m.metadata.get("input_vars", ["tau"])
        sim   = make_ode_simulator(rhs, params, state_v, input_v,
                                   highest_deriv_var="theta_ddot")
        p_arr = np.array([m.parameters.get(p, 1.0) for p in params])
    return sim(p_arr, t, u_arr)

u_flat = u_true[0][:len(t_v)]
y_wb   = _sim_model(wb_fitted_id, t_v, u_flat)
y_gb   = _sim_model(dossier_gb.artifacts.current_model_id, t_v, u_flat)
y_surr = _sim_model(dossier_surr.artifacts.current_model_id, t_v, u_flat)
y_ref  = y_true[0][:len(t_v)]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

axes[0].plot(t_v, y_ref,  color="white",   lw=2.5, label="True plant")
axes[0].plot(t_v, y_wb,   color="#f72585", lw=1.5, ls="--", label="White-box (no Coulomb)")
axes[0].plot(t_v, y_gb,   color="#ffd166", lw=1.5, ls="-.", label="Grey-box (K_c fixed)")
axes[0].plot(t_v, y_surr, color="#7bed9f", lw=1.5, ls=":",  label="Surrogate (GP)")
axes[0].set(ylabel="θ [rad]", title="Model comparison on validation trajectory")
axes[0].legend(loc="upper right"); axes[0].grid(True, alpha=0.3)

axes[1].plot(t_v, y_ref - y_wb,   color="#f72585", label=f"WB error  RMSE={np.sqrt(np.nanmean((y_ref-y_wb)**2)):.3f}")
axes[1].plot(t_v, y_ref - y_gb,   color="#ffd166", label=f"GB error  RMSE={np.sqrt(np.nanmean((y_ref-y_gb)**2)):.3f}")
axes[1].plot(t_v, y_ref - y_surr, color="#7bed9f", label=f"Surr error RMSE={np.sqrt(np.nanmean((y_ref-y_surr)**2)):.3f}")
axes[1].axhline(0, color="white", lw=0.8, ls="--")
axes[1].set(ylabel="Residual θ [rad]", xlabel="Time [s]")
axes[1].legend(loc="upper right"); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 7  Phase portrait

In [ ]:
from agents.estimator import _estimate_velocity

theta_dot_est = _estimate_velocity(t_v, y_ref)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(y_ref, theta_dot_est, c=t_v, cmap="plasma", s=8, alpha=0.8)
plt.colorbar(sc, ax=ax, label="Time [s]")
ax.set(xlabel="θ [rad]", ylabel="θ̇ [rad/s]", title="Phase portrait (true plant)")
ax.axhline(0, color="white", lw=0.5, ls="--")
ax.axvline(0, color="white", lw=0.5, ls="--")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 8  Parameter accuracy summary

In [ ]:
import pandas as pd

def pct(est, true):
    return f"{100 * abs(est - true) / true:.1f} %" if true else "—"

wb_p = wb_fitted.parameters
gb_p = gb_model.parameters

rows = [
    ("K_in",  K_IN_TRUE,  wb_p.get("K_in",  None), gb_p.get("K_in",  None)),
    ("tau_d", TAU_D_TRUE, wb_p.get("tau_d", None), gb_p.get("tau_d", None)),
    ("K_g",   K_G_TRUE,   wb_p.get("K_g",   None), gb_p.get("K_g",   None)),
]
print(f"{'Param':<8} {'True':>8} {'WB fit':>10} {'WB err':>8} {'GB fit':>10} {'GB err':>8}")
print("-" * 58)
for name, true, wb, gb in rows:
    wb_s = f"{wb:.4f}" if wb is not None else "—"
    gb_s = f"{gb:.4f}" if gb is not None else "—"
    print(f"{name:<8} {true:>8.4f} {wb_s:>10} {pct(wb,true) if wb else '—':>8} "
          f"{gb_s:>10} {pct(gb,true) if gb else '—':>8}")